# Training environment check (CPU only)


In [ ]:
import importlib.metadata as md, json, os, platform, subprocess, sys
from pathlib import Path
print('python', sys.version.split()[0], platform.platform())
for p in ['torch','transformers','peft','trl','accelerate','bitsandbytes','compressed-tensors','safetensors','datasets','deepspeed',
          'flash-attn','flash_attn','liger-kernel','xformers','vllm','unsloth','optimum','scipy','numpy']:
    try: print(f'{p:20s}', md.version(p))
    except md.PackageNotFoundError: print(f'{p:20s} -- not installed')
try:
    import torch; print('torch cuda available:', torch.cuda.is_available(), '| build cuda:', torch.version.cuda)
except Exception as e: print('torch import failed', e)
print(subprocess.run('df -h /kaggle/working /tmp | cat; free -g | cat; nproc', shell=True, capture_output=True, text=True).stdout)
cfgs = sorted(Path('/kaggle/input').rglob('config.json'))
print('config.json files under /kaggle/input:', [str(c) for c in cfgs][:10])
for c in cfgs:
    if 'unquantized' in str(c).lower() or '31b' in str(c).lower():
        d = c.parent
        print('\n=== model dir', d)
        files = sorted(d.iterdir())
        print({f.name: round(f.stat().st_size/1e9, 2) for f in files if f.is_file()})
        cfg = json.loads(c.read_text())
        txt = cfg.get('text_config', cfg)
        keep = ['architectures','model_type','hidden_size','num_hidden_layers','num_attention_heads','num_key_value_heads','head_dim','intermediate_size',
                'vocab_size','max_position_embeddings','sliding_window','layer_types','torch_dtype','dtype','quantization_config']
        print('top-level keys:', list(cfg.keys()))
        print(json.dumps({k: cfg[k] for k in keep if k in cfg}, indent=1)[:1500])
        print('text_config:', json.dumps({k: txt[k] for k in keep if k in txt}, indent=1)[:2500])
        idx = d / 'model.safetensors.index.json'
        if idx.exists():
            wm = json.loads(idx.read_text())['weight_map']; ks = list(wm)
            print('n tensors', len(ks), '| sample names:', ks[:3], ks[-3:])
            lay = sorted({k for k in ks if '.layers.0.' in k})
            print('layer-0 tensors:', lay)
